# Feature Engineering


In [1]:
import pandas as pd
# Update the path if your cleaned.csv is in a different folder
df = pd.read_csv('cleaned.csv', encoding='utf-8')
print('Shape:', df.shape)
df.head(4)

Shape: (6839, 16)


,Learner SignUp DateTime,Opportunity Id,Opportunity Name,Opportunity Category,Opportunity End Date,First Name,Date of Birth,Gender,Country,Institution Name,Current/Intended Major,Entry created at,Status Description,Status Code,Apply Date,Opportunity Start Date
0,2023-06-14 12:30:35,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,Course,2024-06-29 18:52:39,Faria,2001-12-01,Female,Pakistan,Nwihs,Radiology,2024-11-03 12:01:41,Started,1080,2023-06-14 12:36:09,2022-03-11 18:30:39
1,2023-01-05 05:29:16,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,Course,2024-06-29 18:52:39,Poojitha,2000-08-16,Female,India,SAINT LOUIS,Information Systems,2024-11-03 12:01:41,Started,1080,2023-01-05 06:08:21,2022-03-11 18:30:39
2,2023-08-29 05:20:03,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,Course,2024-06-29 18:52:39,Amrutha Varshini,1999-01-11,Female,United States,Saint Louis University,Information Systems,2024-11-03 12:01:41,Team Allocated,1070,2023-09-10 22:02:42,2022-03-11 18:30:39
3,2023-06-01 15:26:36,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,Course,2024-06-29 18:52:39,Vinay Varshith,2000-04-19,Male,United States,Saint Louis University,Computer Science,2024-11-03 12:01:41,Started,1080,2023-06-01 15:40:10,2022-03-11 18:30:39


### 1) Creating New Features
 - Age of Learner: calculate from Date of Birth
 - Engagement Duration: time between Apply Date and Opportunity Start Date

In [2]:
# parse dates safely (coerce invalid formats to NaT)
df['Date of Birth'] = pd.to_datetime(df.get('Date of Birth'), errors='coerce')
df['Apply Date'] = pd.to_datetime(df.get('Apply Date'), errors='coerce')
df['Opportunity Start Date'] = pd.to_datetime(df.get('Opportunity Start Date'), errors='coerce')

# Age in years: difference between today and Date of Birth
today = pd.Timestamp.today()
df['age'] = (today - df['Date of Birth']).dt.days // 365
# if age is all NaN (DOB not present), try to keep existing 'age' column if available
if df['age'].isna().all() and 'age' in df.columns:
    df['age'] = df['age']

# Engagement days: difference between Opportunity Start Date and Apply Date (days). Use positive days where Start >= Apply.
df['engagement_days'] = -(df['Opportunity Start Date'] - df['Apply Date']).dt.days

print('Created columns: Date of Birth, Apply Date, Opportunity Start Date, age, engagement_days')
print('Records:', len(df))
df[['Date of Birth','age','Apply Date','Opportunity Start Date','engagement_days']].head(4)

Created columns: Date of Birth, Apply Date, Opportunity Start Date, age, engagement_days
Records: 6839


,Date of Birth,age,Apply Date,Opportunity Start Date,engagement_days
0,2001-12-01,23,2023-06-14 12:36:09,2022-03-11 18:30:39,460.0
1,2000-08-16,25,2023-01-05 06:08:21,2022-03-11 18:30:39,300.0
2,1999-01-11,26,2023-09-10 22:02:42,2022-03-11 18:30:39,549.0
3,2000-04-19,25,2023-06-01 15:40:10,2022-03-11 18:30:39,447.0


## 2) Transforming Existing Features
 - Normalize numerical features (age, engagement_days)
 - One-hot encode categorical columns: Gender, Country, Opportunity Category


In [3]:
from sklearn.preprocessing import StandardScaler
num_cols = [c for c in ['age','engagement_days'] if c in df.columns]
if num_cols:
    scaler = StandardScaler()
    df[num_cols] = scaler.fit_transform(df[num_cols].fillna(0))
    print('Normalized:', num_cols)

cat_cols = [c for c in ['Gender','Country','Opportunity Category'] if c in df.columns]
if cat_cols:
    df = pd.get_dummies(df, columns=cat_cols, prefix=[c.replace(' ','_') for c in cat_cols], dummy_na=False)
    print('One-hot encoded:', cat_cols)

print('Resulting shape:', df.shape)
df.head(4)

Normalized: ['age', 'engagement_days']
One-hot encoded: ['Gender', 'Country', 'Opportunity Category']
Resulting shape: (6839, 91)


,Learner SignUp DateTime,Opportunity Id,Opportunity Name,Opportunity End Date,First Name,Date of Birth,Institution Name,Current/Intended Major,Entry created at,Status Description,...,Country_Uzbekistan,Country_Vietnam,Country_Yemen,Country_Zambia,Country_Zimbabwe,Opportunity_Category_Competition,Opportunity_Category_Course,Opportunity_Category_Engagement,Opportunity_Category_Event,Opportunity_Category_Internship
0,2023-06-14 12:30:35,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,2024-06-29 18:52:39,Faria,2001-12-01,Nwihs,Radiology,2024-11-03 12:01:41,Started,...,False,False,False,False,False,False,True,False,False,False
1,2023-01-05 05:29:16,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,2024-06-29 18:52:39,Poojitha,2000-08-16,SAINT LOUIS,Information Systems,2024-11-03 12:01:41,Started,...,False,False,False,False,False,False,True,False,False,False
2,2023-08-29 05:20:03,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,2024-06-29 18:52:39,Amrutha Varshini,1999-01-11,Saint Louis University,Information Systems,2024-11-03 12:01:41,Team Allocated,...,False,False,False,False,False,False,True,False,False,False
3,2023-06-01 15:26:36,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,2024-06-29 18:52:39,Vinay Varshith,2000-04-19,Saint Louis University,Computer Science,2024-11-03 12:01:41,Started,...,False,False,False,False,False,False,True,False,False,False


### 3) Extracting Useful Components
- Date-Based Features: extract month and year from `Learner SignUp DateTime` to capture seasonal trends.
- Opportunity Engagement: compute `time_in_opportunity` (duration) and see correlation with engagement.

In [4]:
# Extract month/year from Learner SignUp DateTime and compute time in opportunity
df['Learner SignUp DateTime'] = pd.to_datetime(df.get('Learner SignUp DateTime'), errors='coerce')
if 'Learner SignUp DateTime' in df.columns:
    df['signup_month'] = df['Learner SignUp DateTime'].dt.month
    df['signup_year'] = df['Learner SignUp DateTime'].dt.year
    print('Added signup_month and signup_year')

# Time in opportunity: difference between Opportunity End Date and Opportunity Start Date (days)
df['Opportunity End Date'] = pd.to_datetime(df.get('Opportunity End Date'), errors='coerce')
df['time_in_opportunity'] = (df['Opportunity End Date'] - df['Opportunity Start Date']).dt.days
print('Added time_in_opportunity (days)')

print('New columns sample:')
df[['Learner SignUp DateTime','signup_month','signup_year','Opportunity Start Date','Opportunity End Date','time_in_opportunity']].head(4)

Added signup_month and signup_year
Added time_in_opportunity (days)
New columns sample:


,Learner SignUp DateTime,signup_month,signup_year,Opportunity Start Date,Opportunity End Date,time_in_opportunity
0,2023-06-14 12:30:35,6,2023,2022-03-11 18:30:39,2024-06-29 18:52:39,841.0
1,2023-01-05 05:29:16,1,2023,2022-03-11 18:30:39,2024-06-29 18:52:39,841.0
2,2023-08-29 05:20:03,8,2023,2022-03-11 18:30:39,2024-06-29 18:52:39,841.0
3,2023-06-01 15:26:36,6,2023,2022-03-11 18:30:39,2024-06-29 18:52:39,841.0


In [5]:
df.dtypes

Learner SignUp DateTime            datetime64[ns]
Opportunity Id                             object
Opportunity Name                           object
Opportunity End Date               datetime64[ns]
First Name                                 object
                                        ...      
Opportunity_Category_Event                   bool
Opportunity_Category_Internship              bool
signup_month                                int32
signup_year                                 int32
time_in_opportunity                       float64
Length: 94, dtype: object

## 4) Combining Features
- Interaction Features: combine Age and Opportunity Duration to capture interaction effects.
- Engagement Scores: build a simple composite score from normalized components.

In [6]:
# list all columns in the dataframe
cols = df.columns.tolist()
# also return the list for interactive use
print(cols)

['Learner SignUp DateTime', 'Opportunity Id', 'Opportunity Name', 'Opportunity End Date', 'First Name', 'Date of Birth', 'Institution Name', 'Current/Intended Major', 'Entry created at', 'Status Description', 'Status Code', 'Apply Date', 'Opportunity Start Date', 'age', 'engagement_days', "Gender_Don't want to specify", 'Gender_Female', 'Gender_Male', 'Gender_Other', 'Country_Afghanistan', 'Country_Algeria', 'Country_American Samoa', 'Country_Australia', 'Country_Azerbaijan', 'Country_Bangladesh', 'Country_Belarus', 'Country_Bhutan', 'Country_Botswana', 'Country_Brazil', 'Country_British Indian Ocean Territory', 'Country_Cameroon', 'Country_Canada', 'Country_China', 'Country_Congo', "Country_Cote d'Ivoire", 'Country_Dominican Republic', 'Country_Egypt', 'Country_Ethiopia', 'Country_Falkland Islands (Malvinas)', 'Country_France', 'Country_Gambia', 'Country_Germany', 'Country_Ghana', 'Country_Honduras', 'Country_India', 'Country_Indonesia', 'Country_Iran  Islamic Republic of Persian Gulf

### Interaction features (append)
We create a few interaction features to capture how Age and Opportunity Duration interact, plus a category-weighted duration.
These include: age_squared, log_duration, age_x_duration, category_weighted_duration, and normalized variants.

In [7]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler
# ensure required columns exist (safe defaults)
if 'age' not in df.columns:
    df['age'] = 0
# prefer any imputed duration column if present
if 'time_in_opportunity' not in df.columns and 'time_in_opportunity_imputed' in df.columns:
    df['time_in_opportunity'] = df['time_in_opportunity_imputed']
if 'time_in_opportunity' not in df.columns:
    df['time_in_opportunity'] = 0
if 'Opportunity Category_numeric' not in df.columns:
    df['Opportunity Category_numeric'] = 0
# Create raw interaction features
df['age_squared'] = pd.to_numeric(df['age'], errors='coerce').fillna(0).astype(float) ** 2
# ensure durations are numeric and finite before log1p
dur = pd.to_numeric(df['time_in_opportunity'], errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0).astype(float)
# clip extreme values to a reasonable range to prevent overflow in scalers
dur = np.clip(dur, a_min=0, a_max=10_000_000)
df['log_duration'] = np.log1p(dur)
df['age_x_duration'] = pd.to_numeric(df['age'], errors='coerce').fillna(0).astype(float) * dur
# category-weighted duration = duration * (1 + normalized category) -- normalize category safely
cat = pd.to_numeric(df['Opportunity Category_numeric'].fillna(0), errors='coerce').astype(float).replace([np.inf, -np.inf], np.nan).fillna(0).values.reshape(-1,1)
# replace inf/nan in cat and clip
cat = np.clip(cat, a_min=0, a_max=1e9)
try:
    cat_norm = MinMaxScaler().fit_transform(cat).ravel() if len(cat) > 0 else np.zeros(len(df))
except Exception:
    # fallback: scale by medians if MinMax fails
    med = np.nanmedian(cat)
    cat_norm = np.nan_to_num(cat.ravel() / (med if med else 1.0))
df['category_weighted_duration'] = dur * (1 + cat_norm)
# normalized variants for modeling/readability
to_norm = ['age_squared','log_duration','age_x_duration','category_weighted_duration']
mm = MinMaxScaler()
# prepare tmp matrix and sanitize: replace inf, large values, fillna with median per column
tmp = df[to_norm].astype(float)
# replace infs and huge values
tmp = tmp.replace([np.inf, -np.inf], np.nan)
for c in tmp.columns:
    col = tmp[c]
    if col.isna().all():
        tmp[c] = 0
    else:
        med = col.median()
        col = col.fillna(med)
        # clip to reasonable bounds based on median and std to avoid overflow
        std = max(col.std(), 0)
        if std == 0:
            low, high = med - abs(med) - 1, med + abs(med) + 1
        else:
            low, high = med - 10 * std, med + 10 * std
        tmp[c] = col.clip(lower=low, upper=high)
# attempt MinMax scaling, fallback to per-column manual minmax if sklearn fails
try:
    scaled = mm.fit_transform(tmp)
except Exception as e:
    # manual per-column min-max scaling with safe guards
    scaled = np.zeros(tmp.shape)
    for i, c in enumerate(tmp.columns):
        col = tmp[c].astype(float).values
        mn = np.nanmin(col)
        mx = np.nanmax(col)
        if np.isfinite(mn) and np.isfinite(mx) and mx > mn:
            scaled[:, i] = (col - mn) / (mx - mn)
        else:
            scaled[:, i] = 0
# assign normalized columns back safely
for i, f in enumerate(to_norm):
    df[f + '_norm'] = scaled[:, i]
# quick diagnostics
print('Interaction features added: ', to_norm)
print('Sample of new features:')
print(df[['time_in_opportunity','age_squared','log_duration','age_x_duration','category_weighted_duration']].head(8).to_string(index=True))
# if engagement_score exists, show correlations with new features
if 'engagement_score' in df.columns:
    print('\nCorrelations with engagement_score:')
    for f in to_norm:
        try:
            print(f, 'corr=', float(df[f].corr(df['engagement_score'])))
        except Exception:
            pass
print('Done — appended interaction features at the notebook end')

Interaction features added:  ['age_squared', 'log_duration', 'age_x_duration', 'category_weighted_duration']
Sample of new features:
   time_in_opportunity  age_squared  log_duration  age_x_duration  category_weighted_duration
0                841.0     0.319633       6.73578     -475.468533                       841.0
1                841.0     0.010986       6.73578      -88.150411                       841.0
2                841.0     0.015739       6.73578      105.508651                       841.0
3                841.0     0.010986       6.73578      -88.150411                       841.0
4                  NaN     0.343397       0.00000        0.000000                         0.0
5                841.0     0.112284       6.73578     -281.809472                       841.0
6                841.0     2.209535       6.73578    -1250.104779                       841.0
7                841.0     2.209535       6.73578    -1250.104779                       841.0
Done — appended inter

## Engagement Scores

How: Develop a composite score by combining features like Time in Opportunity, Age, and Opportunity Category.

Why: A comprehensive engagement score can provide a single metric to evaluate overall engagement levels across different factors.

Engagement Scores:

How: Develop composite scores by combining several features. For example, use a weighted average formula to combine Opportunity Duration, Age, and other relevant features.

Why: Provides a single metric to evaluate overall engagement.

In [8]:
# Simple composite engagement score (min-max each component then weighted average)
import numpy as np

components = ['time_in_opportunity', 'age', 'Opportunity Category_numeric']
weights = [0.5, 0.25, 0.25]
# ensure columns exist and are numeric
for c in components:
    if c not in df.columns:
        df[c] = 0
    df[c] = pd.to_numeric(df[c], errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0).astype(float)
# simple min-max helper
def minmax_col(arr):
    mn = np.nanmin(arr)
    mx = np.nanmax(arr)
    if not np.isfinite(mn) or not np.isfinite(mx) or mx <= mn:
        return np.zeros_like(arr)
    return (arr - mn) / (mx - mn)
# scale components
scaled = np.vstack([minmax_col(df[c].values.astype(float)) for c in components]).T
# weighted average
w = np.array(weights, dtype=float)
w = w / w.sum()
df['engagement_score_composite'] = scaled.dot(w)
print('Computed engagement_score_composite — sample:')
print(df[['engagement_score_composite']].head(8).to_string(index=True))

Computed engagement_score_composite — sample:
   engagement_score_composite
0                    0.580189
1                    0.589623
2                    0.594340
3                    0.589623
4                    0.107901
5                    0.584906
6                    0.561321
7                    0.561321


### 5. Temporal Analysis — Seasonal Patterns

Seasonal Patterns:

How: Use features extracted from Learner SignUp DateTime, such as the month or day of the week, to analyze seasonal engagement trends.

Why: Identifying patterns in engagement based on time can help in scheduling opportunities and targeting specific periods for enhanced participation.

In [9]:
# Temporal analysis: extract month and weekday from Learner SignUp DateTime and show simple summaries
if 'Learner SignUp DateTime' not in df.columns:
    df['Learner SignUp DateTime'] = pd.to_datetime(df.get('Learner SignUp DateTime'), errors='coerce')
else:
    df['Learner SignUp DateTime'] = pd.to_datetime(df['Learner SignUp DateTime'], errors='coerce')

# extract components
df['signup_month'] = df['Learner SignUp DateTime'].dt.month.fillna(0).astype(int)
df['signup_weekday'] = df['Learner SignUp DateTime'].dt.dayofweek.fillna(-1).astype(int)

# show counts per month and weekday and mean engagement_score_composite if present
print('Counts by month:')
print(df['signup_month'].value_counts().sort_index())
print('\nCounts by weekday (0=Mon):')
print(df['signup_weekday'].value_counts().sort_index())

if 'engagement_score_composite' in df.columns:
    print('\nMean engagement_score_composite by month:')
    print(df.groupby('signup_month')['engagement_score_composite'].mean().round(4))
    print('\nMean engagement_score_composite by weekday:')
    print(df.groupby('signup_weekday')['engagement_score_composite'].mean().round(4))

print('\nDone — temporal summaries appended at the notebook end')

Counts by month:
signup_month
1     814
2     761
3     315
4     424
5     536
6     688
7     533
8     898
9     670
10    328
11    278
12    594
Name: count, dtype: int64

Counts by weekday (0=Mon):
signup_weekday
0    1076
1     970
2    1001
3    1141
4    1128
5     772
6     751
Name: count, dtype: int64

Mean engagement_score_composite by month:
signup_month
1     0.1566
2     0.1929
3     0.2983
4     0.2311
5     0.3141
6     0.3455
7     0.2440
8     0.1970
9     0.1897
10    0.2245
11    0.2117
12    0.1864
Name: engagement_score_composite, dtype: float64

Mean engagement_score_composite by weekday:
signup_weekday
0    0.2315
1    0.2342
2    0.2295
3    0.2229
4    0.2349
5    0.2130
6    0.2131
Name: engagement_score_composite, dtype: float64

Done — temporal summaries appended at the notebook end


### Present key features added

- age — Learner age in years (from Date of Birth)
- engagement_days — days between Apply Date and Opportunity Start Date
- time_in_opportunity — days between Opportunity Start Date and Opportunity End Date
- age_squared — age^2 to capture non-linear age effects
- log_duration — log1p of time_in_opportunity (stabilizes large durations)
- age_x_duration — interaction term (age * time_in_opportunity)
- category_weighted_duration — duration adjusted by normalized Opportunity Category numeric
- *_norm columns for interaction features (age_squared_norm, log_duration_norm, age_x_duration_norm, category_weighted_duration_norm)
- engagement_score_composite — simple weighted composite (time_in_opportunity, age, Opportunity Category_numeric)
- signup_month — month extracted from Learner SignUp DateTime
- signup_weekday — weekday extracted from Learner SignUp DateTime (0=Mon)

These were added to support simple modeling and temporal/interaction analysis.

### Examples and dtypes for added features

Below we print example values (head) for the key features added earlier and show their pandas dtypes. We also provide a tidy table (feature | dtype | example_value).

In [11]:
# Show example rows and dtypes for the features we added
features = [
    'engagement_days', 'time_in_opportunity',
    'age_squared','log_duration','age_x_duration','category_weighted_duration',
    'age_squared_norm','log_duration_norm','age_x_duration_norm','category_weighted_duration_norm',
    'engagement_score_composite','signup_month','signup_weekday'
]

existing = [f for f in features if f in df.columns]
print('Existing features (count):', len(existing))
print('\nSample rows (head) for existing features:')
if existing:
    display(df[existing].head(8))
else:
    print('No features from the list are present in the dataframe yet.')

print('\nDtypes:')
dtypes = df[existing].dtypes if existing else pd.Series(dtype='object')
print(dtypes)

# Tidy table: feature, dtype, example value
rows = []
for f in existing:
    sample = df[f].dropna().iloc[0] if df[f].dropna().shape[0] > 0 else None
    rows.append({'feature': f, 'dtype': str(df[f].dtype), 'example_value': sample})

if rows:
    tidy = pd.DataFrame(rows)
    print('\nTidy table:')
    display(tidy)
else:
    print('\nNo tidy table to show (no features present)')

Existing features (count): 13

Sample rows (head) for existing features:


,engagement_days,time_in_opportunity,age_squared,log_duration,age_x_duration,category_weighted_duration,age_squared_norm,log_duration_norm,age_x_duration_norm,category_weighted_duration_norm,engagement_score_composite,signup_month,signup_weekday
0,1.571437,841.0,0.319633,6.73578,-475.468533,841.0,0.009937,1.0,0.245085,1.0,0.580189,6,2
1,0.911797,841.0,0.010986,6.73578,-88.150411,841.0,0.000000,1.0,0.306356,1.0,0.589623,1,3
2,1.938361,841.0,0.015739,6.73578,105.508651,841.0,0.000153,1.0,0.336992,1.0,0.594340,8,1
3,1.517841,841.0,0.010986,6.73578,-88.150411,841.0,0.000000,1.0,0.306356,1.0,0.589623,6,3
4,-0.325027,0.0,0.343397,0.00000,0.000000,0.0,0.010702,0.0,0.320301,0.0,0.107901,2,5
5,1.571437,841.0,0.112284,6.73578,-281.809472,841.0,0.003261,1.0,0.275720,1.0,0.584906,5,2
6,1.728101,841.0,2.209535,6.73578,-1250.104779,841.0,0.070784,1.0,0.122542,1.0,0.561321,7,5
7,1.484859,841.0,2.209535,6.73578,-1250.104779,841.0,0.070784,1.0,0.122542,1.0,0.561321,3,0



Dtypes:
engagement_days                    float64
time_in_opportunity                float64
age_squared                        float64
log_duration                       float64
age_x_duration                     float64
category_weighted_duration         float64
age_squared_norm                   float64
log_duration_norm                  float64
age_x_duration_norm                float64
category_weighted_duration_norm    float64
engagement_score_composite         float64
signup_month                         int64
signup_weekday                       int64
dtype: object

Tidy table:


,feature,dtype,example_value
0,engagement_days,float64,1.571437
1,time_in_opportunity,float64,841.000000
2,age_squared,float64,0.319633
3,log_duration,float64,6.735780
4,age_x_duration,float64,-475.468533
5,category_weighted_duration,float64,841.000000
6,age_squared_norm,float64,0.009937
7,log_duration_norm,float64,1.000000
8,age_x_duration_norm,float64,0.245085
9,category_weighted_duration_norm,float64,1.000000


In [ ]:
from dateutil import parser as dateparser
import pandas as pd
import numpy as np
import re

# --- Recompute `age` from `Date of Birth` robustly ---
if 'age' in df.columns:
    df.drop(columns=['age'], inplace=True, errors='ignore')

def parse_dob(val):
    """Parse a single Date of Birth string robustly."""
    if pd.isna(val):
        return pd.NaT

    s = str(val).strip()
    # Normalize separators (replace ., _, \, or space with /)
    s = re.sub(r'[\.\-_\s]+', '/', s)

    # Try common explicit formats first
    fmts = [
        '%d/%m/%Y', '%m/%d/%Y', '%Y/%m/%d',
        '%d-%m-%Y', '%Y-%m-%d', '%b %d, %Y'
    ]
    for f in fmts:
        try:
            return pd.to_datetime(s, format=f, errors='raise')
        except Exception:
            continue

    # Fallback: use dateutil parser (first month-first, then day-first)
    for dayfirst in [False, True]:
        try:
            return pd.to_datetime(dateparser.parse(s, fuzzy=True, dayfirst=dayfirst))
        except Exception:
            continue

    return pd.NaT

if 'Date of Birth' in df.columns:
    dob_raw = df['Date of Birth'].astype(str).replace(['nan', 'NaN', 'None'], np.nan)
    
    # Fast parse attempt
    parsed = pd.to_datetime(dob_raw, errors='coerce')

    # Retry failed parses with robust function
    mask = parsed.isna() & dob_raw.notna()
    if mask.any():
        parsed.loc[mask] = dob_raw[mask].apply(parse_dob)

    # Final clean date column
    df['Date of Birth'] = parsed.dt.normalize()

    # Compute age in years (accounting for leap years)
    today = pd.Timestamp.today().normalize()
    df['age'] = np.floor((today - df['Date of Birth']).dt.days / 365.25)
    df['age'] = pd.to_numeric(df['age'], errors='coerce')

    print(f" Recomputed 'age' for {df['age'].notna().sum()} of {len(df)} rows.")
else:
    print(" No 'Date of Birth' column found — cannot compute age.")

✅ Recomputed 'age' for 6839 of 6839 rows.


In [13]:
df.head(4)

,Learner SignUp DateTime,Opportunity Id,Opportunity Name,Opportunity End Date,First Name,Date of Birth,Institution Name,Current/Intended Major,Entry created at,Status Description,...,log_duration,age_x_duration,category_weighted_duration,age_squared_norm,log_duration_norm,age_x_duration_norm,category_weighted_duration_norm,engagement_score_composite,signup_weekday,age
0,2023-06-14 12:30:35,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,2024-06-29 18:52:39,Faria,2001-12-01,Nwihs,Radiology,2024-11-03 12:01:41,Started,...,6.73578,-475.468533,841.0,0.009937,1.0,0.245085,1.0,0.580189,2,23.0
1,2023-01-05 05:29:16,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,2024-06-29 18:52:39,Poojitha,2000-08-16,SAINT LOUIS,Information Systems,2024-11-03 12:01:41,Started,...,6.73578,-88.150411,841.0,0.000000,1.0,0.306356,1.0,0.589623,3,25.0
2,2023-08-29 05:20:03,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,2024-06-29 18:52:39,Amrutha Varshini,1999-01-11,Saint Louis University,Information Systems,2024-11-03 12:01:41,Team Allocated,...,6.73578,105.508651,841.0,0.000153,1.0,0.336992,1.0,0.594340,1,26.0
3,2023-06-01 15:26:36,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,2024-06-29 18:52:39,Vinay Varshith,2000-04-19,Saint Louis University,Computer Science,2024-11-03 12:01:41,Started,...,6.73578,-88.150411,841.0,0.000000,1.0,0.306356,1.0,0.589623,3,25.0


In [14]:
# Export the dataframe 'df' to an Excel file
output_path = "Feature_Engineered_Data.xlsx"

# index=False → removes the dataframe index column
# engine='openpyxl' ensures correct Excel writing
df.to_excel(output_path, index=False, engine='openpyxl')

print(f"✅ DataFrame exported successfully to '{output_path}'")

✅ DataFrame exported successfully to 'Feature_Engineered_Data.xlsx'
